## Imports

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  #load all the environment variables

#https://python.langchain.com/docs/integrations/text_embedding/

True

In [3]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/opt/homebrew/Caskroom/miniconda/base/envs/agentic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
text = "This is a test text for embedding."
# Get the embedding for the text
embedding = embeddings.embed_query(text)
print(embedding)
print(len(embedding))  # Should print the dimension of the embedding vector, typically 384 for this model

[-0.021922197192907333, 0.03704843297600746, 0.03152918070554733, 0.023410942405462265, 0.04594523087143898, 0.02833828516304493, 0.008210246451199055, -0.031240839511156082, 0.016568470746278763, -0.01384261529892683, 0.06609421223402023, 0.032968681305646896, 0.03136320412158966, 0.014481732621788979, -0.05385420098900795, 0.034502141177654266, 0.08494751155376434, -0.019580809399485588, -0.02509688027203083, -0.0024577677249908447, 0.01875726878643036, 0.0854223445057869, 0.07826600223779678, -0.04895962029695511, 0.019047847017645836, 0.018522925674915314, -0.0968204066157341, 0.06553623825311661, 0.07570167630910873, -0.01970900036394596, 0.04422162473201752, -0.0011081310221925378, 0.049433041363954544, 0.08979321271181107, 0.0496470108628273, 0.05496825650334358, 0.018358051776885986, 0.018343420699238777, 0.0017473010811954737, 0.054756633937358856, 0.017944850027561188, -0.06672463566064835, 0.030171748250722885, 0.057085778564214706, 0.009166388772428036, -0.03899941220879555

## Cosine Similarity Search in Action

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
documents = [
    "what is capital of USA?",
    "who is the president of USA?",
    "who is the prime minister of India?",
]

my_query = "Narendra Modi is the prime minister of India."

In [11]:
document_embeddings = embeddings.embed_documents(documents)
query_embedding = embeddings.embed_query(my_query)

In [12]:
cosine_similarities = cosine_similarity([query_embedding], document_embeddings)
print(cosine_similarities)

[[0.06941345 0.33584689 0.7616893 ]]


Similarity search measure the two sentances how much these sentance are similar represent in numbers

## Euclidean distance in action

In [13]:
from sklearn.metrics.pairwise import euclidean_distances

In [14]:
euclidean_distances = euclidean_distances([query_embedding], document_embeddings)
print(euclidean_distances)

[[1.36424826 1.1525217  0.69037774]]


euclidean distance measure distance btw two sentances and smallest distance show two sentances are more similar

## Comparison

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


## Vectore store in action

In [12]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


In [21]:
faiss_index = faiss.IndexFlatL2(len(embedding))  # Create a FAISS index for L2 distance
faiss_index_cosine = faiss.IndexFlatIP(len(embedding))  # Create a FAISS index for cosine similarity

In [25]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [27]:
documents = ["AI is future", "AI is powerful", "Dogs are cute"]

In [28]:
vector_store.add_texts(texts=documents)

['f89a050d-21fe-40ea-b32d-7f9d73e2c755',
 '8ba4641a-d0bf-47ce-8663-898787cd3e1a',
 'c567aabc-b674-43ab-9a33-86e25b2cef72']

In [30]:
vector_store.index_to_docstore_id

{0: 'f89a050d-21fe-40ea-b32d-7f9d73e2c755',
 1: '8ba4641a-d0bf-47ce-8663-898787cd3e1a',
 2: 'c567aabc-b674-43ab-9a33-86e25b2cef72'}

In [32]:
vector_store.similarity_search("Tell me about AI", k=2)

[Document(id='8ba4641a-d0bf-47ce-8663-898787cd3e1a', metadata={}, page_content='AI is powerful'),
 Document(id='f89a050d-21fe-40ea-b32d-7f9d73e2c755', metadata={}, page_content='AI is future')]

In [10]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [17]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [18]:
vector_store.add_documents(documents=documents)

['181c770a-04e3-495c-855d-2297d79205ed',
 '30d77459-951f-4568-a338-9d92da8788d4',
 'a427a540-7e90-4c0f-8d01-d9bdf0deb064',
 '62dec6e3-691d-4e97-a590-6ce7700d49b9',
 'd43da83e-e8dd-4977-810d-47e429f7a44b',
 '7f97b1d8-b8e9-49a4-9e74-bfd8331db4ea',
 '1d0f29f8-df9a-47cd-a1c6-991e03919189',
 '6a1b36e3-0080-4cc9-b5c6-d5a79dbf3a68',
 '4d76210a-980b-4a60-8911-f8343692c19e',
 '9023d69b-cf08-4894-aaf2-3c514e562ed8']

In [19]:
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2 #hyperparameter

)

[Document(id='a427a540-7e90-4c0f-8d01-d9bdf0deb064', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='6a1b36e3-0080-4cc9-b5c6-d5a79dbf3a68', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [39]:
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2, #hyperparameter
    filter={"source": {"$eq":"tweet"}}  # Filter by metadata
)

[Document(id='f7d07d92-21ea-453c-a544-01ca2588aa04', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='6ef690f9-fc16-4741-b6e1-dede0fca8d10', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [20]:
retriever =vector_store.as_retriever(
    search_kwargs={
        "k": 2,  # Number of documents to return
    }
)

In [45]:
retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

[Document(id='f7d07d92-21ea-453c-a544-01ca2588aa04', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='6ef690f9-fc16-4741-b6e1-dede0fca8d10', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [ ]:
# inmemory(server)
# ondisk(server)
# cloud(yet to discuss)

In [46]:
vector_store.save_local("today's class faiss index")

In [47]:
new_vector_store=FAISS.load_local(
  "today's class faiss index",embeddings ,allow_dangerous_deserialization=True
)

In [48]:
new_vector_store.similarity_search("langchain")

[Document(id='f7d07d92-21ea-453c-a544-01ca2588aa04', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='6ef690f9-fc16-4741-b6e1-dede0fca8d10', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='0eb4e2c3-ad70-46ec-adbd-e0a862070970', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='b574a12a-4ebf-4e74-b93c-a96b03d876b2', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

In [5]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [6]:
import pprint
pprint.pprint(prompt)

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
context(retriever),prompt(hub),model(google),parser(langchain)

In [8]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [24]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT_ASSIGNMENT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"


In [26]:
from langchain_groq import ChatGroq

# Initialize the model
model = ChatGroq(
    model="qwen-qwq-32b",
    temperature=0.0,
)



In [27]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [28]:
rag_chain.invoke("what is llama model?")

'\n<think>\nOkay, the user is asking about the Llama model. Let me check the provided contexts. The first context mentions "Building an exciting new project with LangChain" but doesn\'t specify what Llama is. The second context is about the weather, which is irrelevant. Since there\'s no information about Llama in the given contexts, I should respond that I don\'t know. I need to make sure not to use any external knowledge here. The answer should be straightforward.\n</think>\n\nI don\'t know the answer. The provided contexts do not contain information about the Llama model.'